# Load Olist CSV files from GCS into Cloud SQL PostgreSQL

This notebook loads Olist CSV files from a Google Cloud Storage batch folder into Cloud SQL for PostgreSQL.

Configuration split:

- `.env` keeps stable database connection values and secrets only.
- YAML config controls the batch, GCS prefix, load mode, and CSV-to-table mappings.

This supports separate folders for initial and incremental loads, for example:

```text
Initial Load/
Incremental Load/2026-06-05/
```


In [ ]:
# Import packages

import io
import os
import re
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
import yaml
from dotenv import load_dotenv
from google.cloud import storage
from google.cloud.sql.connector import Connector
from sqlalchemy import create_engine, text


In [ ]:
# ------------------------------------------------------------
# Load .env values
# ------------------------------------------------------------
# .env should contain stable connection values and secrets only, for example:
# INSTANCE_CONNECTION_NAME=project:region:cloudsql-instance
# DB_USER=postgres
# DB_PASS=your-password
# DB_NAME=olist
# GCP_PROJECT_ID=your-gcp-project-id          # optional but useful
# CONFIG_PATH=../src/olist_load_config.yaml   # optional override

ENV_PATH = Path.cwd().parent / "src/.env"
if not ENV_PATH.exists():
    raise FileNotFoundError(f".env file not found: {ENV_PATH}")

load_dotenv(dotenv_path=ENV_PATH, override=True)

INSTANCE_CONNECTION_NAME = os.getenv("INSTANCE_CONNECTION_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_NAME = os.getenv("DB_NAME")
GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID")

required_env_vars = {
    "INSTANCE_CONNECTION_NAME": INSTANCE_CONNECTION_NAME,
    "DB_USER": DB_USER,
    "DB_PASS": DB_PASS,
    "DB_NAME": DB_NAME,
}

missing_env_vars = [key for key, value in required_env_vars.items() if not value]
if missing_env_vars:
    raise ValueError(f"Missing required .env values: {missing_env_vars}")

print("Required .env values are set.")
print(f"GCP_PROJECT_ID: {GCP_PROJECT_ID}")
print(f"Using Cloud SQL instance: {INSTANCE_CONNECTION_NAME}")
print(f"Database user: {DB_USER}")
print(f"Database name: {DB_NAME}")
print(f"Database password: {'*' * len(DB_PASS)}")


In [ ]:
# ------------------------------------------------------------
# Load YAML batch config
# ------------------------------------------------------------
# CONFIG_PATH can be set in .env. Otherwise this notebook expects:
# ../src/olist_load_config.yaml

CONFIG_PATH = Path(os.getenv("CONFIG_PATH", Path.cwd().parent / "src/olist_load_config.yaml"))
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"YAML config file not found: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config: Dict[str, Any] = yaml.safe_load(f)

# Expected YAML structure:
# gcp:
#   project_id: your-project-id
# gcs:
#   bucket: m2_olin_raw_johnphs
# postgres:
#   schema: oltp
# batch:
#   batch_name: initial_load
#   gcs_prefix: "Initial Load/"
#   load_mode: replace
# csv_mappings:
#   - file: olist_customers_dataset.csv
#     table: olist_customers

GCP_PROJECT_ID = config.get("gcp", {}).get("project_id", GCP_PROJECT_ID) or GCP_PROJECT_ID
GCS_BUCKET_NAME = config["gcs"]["bucket"]
POSTGRES_SCHEMA = config.get("postgres", {}).get("schema", "oltp")

batch_config = config["batch"]
BATCH_NAME = batch_config["batch_name"]
GCS_PREFIX = batch_config["gcs_prefix"]
LOAD_MODE = batch_config.get("load_mode", "append")

CSV_MAPPINGS: List[Dict[str, str]] = config["csv_mappings"]

allowed_load_modes = {"replace", "append", "fail"}
if LOAD_MODE not in allowed_load_modes:
    raise ValueError(f"Invalid load_mode: {LOAD_MODE}. Use one of {sorted(allowed_load_modes)}")

print("YAML config loaded.")
print(f"Config path: {CONFIG_PATH}")
print(f"GCS bucket: {GCS_BUCKET_NAME}")
print(f"Batch name: {BATCH_NAME}")
print(f"GCS prefix: {GCS_PREFIX}")
print(f"Load mode: {LOAD_MODE}")
print(f"PostgreSQL schema: {POSTGRES_SCHEMA}")
print(f"CSV mappings: {len(CSV_MAPPINGS)} files")


In [ ]:
# ------------------------------------------------------------
# Validation helpers
# ------------------------------------------------------------

def validate_identifier(identifier: str, label: str) -> None:
    """Keep schema/table names safe for quoted PostgreSQL identifiers."""
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(
            f"Invalid {label}: {identifier!r}. Use letters, numbers, and underscores only; "
            "the first character must be a letter or underscore."
        )


def build_gcs_path(prefix: str, filename: str) -> str:
    return f"{prefix.rstrip('/')}/{filename}"


validate_identifier(POSTGRES_SCHEMA, "PostgreSQL schema")

for mapping in CSV_MAPPINGS:
    if "file" not in mapping or "table" not in mapping:
        raise ValueError(f"Each csv_mappings entry must contain file and table: {mapping}")
    validate_identifier(mapping["table"], "table name")

print("Config validation passed.")


In [ ]:
# ------------------------------------------------------------
# Cloud SQL PostgreSQL connection
# ------------------------------------------------------------
connector = Connector()


def getconn():
    return connector.connect(
        INSTANCE_CONNECTION_NAME,
        "pg8000",
        user=DB_USER,
        password=DB_PASS,
        db=DB_NAME,
    )


engine = create_engine(
    "postgresql+pg8000://",
    creator=getconn,
)


# ------------------------------------------------------------
# Create schema if it does not exist
# ------------------------------------------------------------
def create_schema_if_not_exists() -> None:
    with engine.begin() as conn:
        conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{POSTGRES_SCHEMA}"'))
    print(f"Schema ready: {POSTGRES_SCHEMA}")


In [ ]:
# ------------------------------------------------------------
# Load one CSV from GCS into PostgreSQL
# ------------------------------------------------------------
def get_safe_chunksize(num_columns: int, max_bind_params: int = 60_000) -> int:
    """Avoid PostgreSQL bind-parameter limit when using pandas to_sql(method='multi')."""
    return max(1, max_bind_params // max(1, num_columns))


def load_csv_from_gcs_to_postgres(
    storage_client: storage.Client,
    filename: str,
    table_name: str,
    if_exists: str,
) -> None:
    bucket = storage_client.bucket(GCS_BUCKET_NAME)
    blob_path = build_gcs_path(GCS_PREFIX, filename)
    source_gcs_path = f"gs://{GCS_BUCKET_NAME}/{blob_path}"
    blob = bucket.blob(blob_path)

    if not blob.exists():
        raise FileNotFoundError(f"File not found: {source_gcs_path}")

    print(f"
Reading {source_gcs_path}")

    csv_bytes = blob.download_as_bytes()
    df = pd.read_csv(io.BytesIO(csv_bytes))

    # Add ingestion metadata for lineage and soft-delete compatibility.
    now_utc = pd.Timestamp.now(tz="UTC")
    df["created_at"] = now_utc
    df["updated_at"] = now_utc
    df["is_deleted"] = False
    df["source_file"] = filename
    df["source_gcs_path"] = source_gcs_path
    df["batch_name"] = BATCH_NAME

    chunksize = get_safe_chunksize(len(df.columns))

    print(f"Rows: {len(df):,}")
    print(f"Columns including metadata: {len(df.columns):,}")
    print(f"Safe chunksize: {chunksize:,}")
    print(f"Loading into {POSTGRES_SCHEMA}.{table_name} with if_exists='{if_exists}'")

    df.to_sql(
        table_name,
        engine,
        schema=POSTGRES_SCHEMA,
        if_exists=if_exists,
        index=False,
        chunksize=chunksize,
        method="multi",
    )

    print(f"Loaded {POSTGRES_SCHEMA}.{table_name}")

    with engine.begin() as conn:
        print(f"Analyzing {POSTGRES_SCHEMA}.{table_name} for query performance...")
        conn.execute(text(f'ANALYZE "{POSTGRES_SCHEMA}"."{table_name}"'))

        table_stats_sql = text("""
            SELECT
                schemaname,
                relname AS table_name,
                n_live_tup AS estimated_live_rows,
                n_dead_tup AS estimated_dead_rows,
                last_analyze,
                last_autoanalyze
            FROM pg_stat_user_tables
            WHERE schemaname = :schema_name
              AND relname = :table_name
        """)

        table_stats = pd.read_sql(
            table_stats_sql,
            conn,
            params={"schema_name": POSTGRES_SCHEMA, "table_name": table_name},
        )

        print("
PostgreSQL table statistics:")
        display(table_stats)


In [ ]:
# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
def main() -> None:
    create_schema_if_not_exists()

    storage_client = storage.Client(project=GCP_PROJECT_ID) if GCP_PROJECT_ID else storage.Client()

    for mapping in CSV_MAPPINGS:
        filename = mapping["file"]
        table_name = mapping["table"]
        active = mapping.get("active", True)

        if not active:
            print(f"
Skipping inactive mapping: {filename} -> {table_name}")
            continue

        print(f"
Processing file: {filename} -> table: {table_name}")
        load_csv_from_gcs_to_postgres(
            storage_client=storage_client,
            filename=filename,
            table_name=table_name,
            if_exists=LOAD_MODE,
        )

    connector.close()
    print("
All configured CSV files loaded successfully.")


if __name__ == "__main__":
    main()
